## Revisiting Sequence Models: RNNs, LSTMs, and Their Limits

Hello there! Welcome to the first lesson of our course "Sequence Models & The Dawn of Attention". This initial lesson marks the beginning of an exciting journey that will take us from traditional sequence modeling approaches all the way to the cutting-edge Transformer architecture that powers today's most advanced AI systems.

Before we dive into our exploration of sequence models, let's establish what we expect you to bring to this course path. We assume you have solid foundations in Python programming, basic machine learning concepts, and PyTorch fundamentals, including tensor operations and neural network construction. Familiarity with basic linear algebra and calculus will also serve you well as we work through mathematical concepts.

Our learning path consists of four comprehensive courses designed to give you deep expertise in Transformer architectures. In this first Course, we'll explore why traditional RNNs and LSTMs struggle with long sequences and build attention mechanisms from scratch, mastering the query-key-value paradigm. Next, in Course 2 we'll systematically construct the complete Transformer architecture, creating reusable components such as multi-head attention modules, feed-forward networks, and positional encodings. By Course 3 we'll be able to combine everything into a complete Transformer model, implementing autoregressive training with teacher forcing and exploring different decoding and inference strategies. Finally, in the last Course we'll harness the powerful Hugging Face ecosystem to work with pre-trained models and fine-tune them for real-world applications.

By the end of this path, you'll have gained a profound understanding of how Transformers work under the hood and the practical skills to apply them effectively. Today, in our first lesson, we begin by revisiting sequence models to understand their limitations and why the AI community needed something better. Let's get started!

---

### The Evolution of Sequence Modeling

Sequential data surrounds us everywhere: the words in this sentence follow a specific order, stock prices change over time, and DNA sequences encode genetic information through their arrangement. Traditional feedforward neural networks process each input independently, but sequence modeling requires understanding how current elements relate to previous ones.

The challenge lies in **temporal dependencies**: understanding that "the cat sat on the" should be followed by something like "mat" requires remembering context from earlier in the sequence. Early approaches like n-grams could only look back a fixed number of steps, severely limiting their ability to capture long-range relationships. This limitation sparked the development of recurrent architectures that could theoretically maintain information across arbitrary sequence lengths.

However, as we'll discover today, even sophisticated recurrent models face fundamental challenges when sequences grow long. This limitation will motivate the development of attention mechanisms and, later, of the Transformer architecture.

---

### Brief Recap: RNNs and LSTMs

Recurrent Neural Networks introduced the revolutionary concept of **hidden states** that carry information from one time step to the next. At each position, an RNN combines the current input with the previous hidden state:

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b_t)$$

Here, $h_t$ represents the current hidden state, $h_{t-1}$ is the previous hidden state, $x_t$ is the current input, $W_h$ and $W_x$ are learned weight matrices for the hidden state and input respectively, $b$ is a bias term, and $\tanh$ is the activation function. This simple idea enables the network to maintain a form of memory.

LSTMs enhanced this concept by introducing **gating mechanisms** to control information flow:

- The **forget gate** decides what to discard from memory;
- The **input gate** determines what new information to store;
- The **output gate** controls what parts of the memory to reveal.

These gates help LSTMs maintain relevant information over longer sequences than basic RNNs.

The key insight behind LSTMs is their **cell state**: a separate memory channel that flows through the network with minimal transformations. Gates can add or remove information from this cell state, but they don't fundamentally alter its structure at each step. This design helps preserve gradients during backpropagation, enabling the model to learn longer-range dependencies than traditional RNNs.

---

### Understanding Our Long-Range Dependency Task

Now, let's examine the specific challenge we'll use to demonstrate sequence model limitations. Our task involves predicting the **first element** of a sequence based on observing the entire sequence, which creates a long-range dependency problem as sequences grow longer:

```python
import torch

def generate_sequences(seq_length, num_samples=100):
    """Generate sequences where target is the first element of the sequence"""
    torch.manual_seed(0)  # For reproducibility
    sequences = torch.randn(num_samples, seq_length, 1)
    targets = sequences[:, 0, 0]  # First element of each sequence
    return sequences, targets.unsqueeze(1)
```

This function creates our experimental setup. We generate random sequences of specified lengths, where each sequence contains real-valued numbers. The crucial aspect is that our target is always the very first element of each sequence; that is, as sequences get longer, the model must remember this initial value while processing many intervening elements.

Notice the tensor shapes: `sequences` has dimensions `(num_samples, seq_length, 1)`, representing batch size, sequence length, and feature dimension. The `targets` tensor becomes `(num_samples, 1)` after we extract and reshape the first elements. This structure follows standard NLP conventions, where we often work with three-dimensional tensors for sequential data.

---

### Building Our LSTM Model

Our LSTM implementation follows PyTorch conventions while keeping the architecture simple enough to clearly observe performance patterns across different sequence lengths:

```python
import torch.nn as nn

class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize hidden state and cell state
        h0 = torch.zeros(1, x.size(0), self.hidden_size)
        c0 = torch.zeros(1, x.size(0), self.hidden_size)

        # Forward propagate LSTM
        out, (hidden, cell) = self.lstm(x, (h0, c0))

        # Decode the hidden state of the last time step
        out = self.linear(out[:, -1, :])
        return out, hidden
```

Our model architecture consists of a single LSTM layer followed by a linear output layer. The `batch_first=True` parameter ensures our input tensors follow the `(batch, sequence, features)` convention. We initialize hidden and cell states to zeros for each forward pass, giving each sequence a fresh start.

The key decision here is using `out[:, -1, :]`, that is, we only use the final time step's hidden state for prediction. This forces the LSTM to compress all relevant information from the entire sequence into this final hidden state, creating the exact challenge we want to study.

---

### Training and Evaluation Framework

Our training function implements a standard supervised learning loop optimized for observing performance degradation patterns.

```python
import torch.optim as optim

def train_and_evaluate(model, sequences, targets, epochs=50):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs, _ = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    # Return final loss
    with torch.no_grad():
        final_outputs, _ = model(sequences)
        final_loss = criterion(final_outputs, targets)
    return final_loss.item()
```

We use **Mean Squared Error** as our loss function since we're predicting continuous values. The Adam optimizer provides stable convergence with a learning rate that works well across different sequence lengths. After training, we evaluate the model one final time with gradients disabled to get a clean measure of final performance.

The function returns the final loss value, which serves as our metric for comparing performance across different sequence lengths. Higher loss values indicate the model struggles more with that particular sequence length.

---

### Systematic Performance Analysis

Now we can systematically test how LSTM performance degrades as sequences grow longer, revealing the fundamental limitations we'll address with attention mechanisms.

```python
def main():
    # Test different sequence lengths
    sequence_lengths = [5, 10, 20, 30, 50, 100]
    results = {}

    print("Testing LSTM performance on different sequence lengths...")

    for seq_len in sequence_lengths:
        # Generate data and create model
        sequences, targets = generate_sequences(seq_len)
        model = SimpleLSTM(input_size=1, hidden_size=16, output_size=1)

        # Train and evaluate
        final_loss = train_and_evaluate(model, sequences, targets)
        results[seq_len] = final_loss

        print(f"Sequence length {seq_len}: Final loss = {final_loss:.4f}")
```

Our experiment systematically tests sequence lengths from 5 to 100 elements. For each length, we create fresh data according to the current `seq_len`, as well as a new model to ensure a fair comparison. The results reveal a clear pattern:

```text
Testing LSTM performance on different sequence lengths...
Sequence length 5: Final loss = 0.0854
Sequence length 10: Final loss = 0.0651
Sequence length 20: Final loss = 0.6047
Sequence length 30: Final loss = 0.7171
Sequence length 50: Final loss = 0.5680
Sequence length 100: Final loss = 0.9084
```

The pattern is striking: performance remains strong for very short sequences but degrades substantially as sequences grow longer, with significant deterioration evident beyond length 10. We can also plot these values to make this trend even clearer (a simple plot showing LSTM performance degradation with increasing sequence length).

This performance degradation demonstrates the **long-range dependency problem** that motivated the development of attention mechanisms: despite LSTMs' theoretical ability to maintain long-term memory, practical limitations emerge when sequences grow beyond certain lengths.

---

### Conclusion and Next Steps

Today, we revisited the foundations of sequence modeling and observed firsthand why RNNs and LSTMs, despite their innovations, face fundamental challenges with long sequences. The experiment we conducted demonstrates how even sophisticated gating mechanisms struggle to maintain relevant information across extended sequences. This limitation isn't merely academic — it affects real-world applications where understanding long-range dependencies is crucial, from document comprehension to protein sequence analysis.

The degradation we observed motivates the need for alternative architectures that can efficiently connect distant elements in sequences. In our next lesson, we'll begin exploring attention mechanisms: the breakthrough that addresses these limitations by allowing models to directly access any position in a sequence. This innovation will pave our way toward understanding the Transformer architecture that revolutionized modern AI. But now, it's time for some practice!

## Building Your First LSTM Model

Now that you've explored the LSTM architecture and its components in the lesson, it's time to implement your own LSTM model. The provided code is missing the complete `SimpleLSTM` class, which you need to build from scratch.

---

### Your Task

Your implementation should include two main methods:

- `__init__`: set up the LSTM layer and the linear output layer.
- `forward`: process input sequences and return predictions.

The rest of the code handles training and testing your LSTM on sequences of different lengths, demonstrating how performance degrades as sequences get longer. This hands-on implementation will give you a solid understanding of LSTM fundamentals.

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


# TODO: Implement the SimpleLSTM class
# This class should inherit from nn.Module and include:
# - __init__ method that sets up the LSTM layer and linear output layer
# - forward method that processes input sequences and returns predictions

def generate_sequences(seq_length, num_samples=100):
    """Generate sequences where target is the first element of the sequence"""
    torch.manual_seed(0)  # For reproducibility
    sequences = torch.randn(num_samples, seq_length, 1)
    targets = sequences[:, 0, 0]  # First element of each sequence
    return sequences, targets.unsqueeze(1)

def train_and_evaluate(model, sequences, targets, epochs=50):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs, _ = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    
    # Return final loss
    with torch.no_grad():
        final_outputs, _ = model(sequences)
        final_loss = criterion(final_outputs, targets)
    return final_loss.item()

def main():
    # Test different sequence lengths
    sequence_lengths = [5, 10, 20, 30, 50]
    results = {}
    
    print("Testing LSTM performance on different sequence lengths...")
    
    for seq_len in sequence_lengths:
        # Generate data and create model
        sequences, targets = generate_sequences(seq_len)
        model = SimpleLSTM(input_size=1, hidden_size=16, output_size=1)
        
        # Train and evaluate
        final_loss = train_and_evaluate(model, sequences, targets)
        results[seq_len] = final_loss
        
        print(f"Sequence length {seq_len}: Final loss = {final_loss:.4f}")
    
    # Visualize performance degradation
    plt.figure(figsize=(10, 6))
    lengths = list(results.keys())
    losses = list(results.values())
    
    plt.plot(lengths, losses, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Sequence Length')
    plt.ylabel('Final Loss (MSE)')
    plt.title('LSTM Performance Degradation with Increasing Sequence Length')
    plt.grid(True, alpha=0.3)
    plt.show()

if __name__ == "__main__":
    main()

```

```
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize hidden state and cell state
        h0 = torch.zeros(1, x.size(0), self.hidden_size)
        c0 = torch.zeros(1, x.size(0), self.hidden_size)

        # Forward propagate LSTM
        out, (hidden, cell) = self.lstm(x, (h0, c0))

        # Decode the hidden state of the last time step
        out = self.linear(out[:, -1, :])
        return out, hidden


def generate_sequences(seq_length, num_samples=100):
    """Generate sequences where target is the first element of the sequence"""
    torch.manual_seed(0)  # For reproducibility
    sequences = torch.randn(num_samples, seq_length, 1)
    targets = sequences[:, 0, 0]  # First element of each sequence
    return sequences, targets.unsqueeze(1)


def train_and_evaluate(model, sequences, targets, epochs=50):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs, _ = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    # Return final loss
    with torch.no_grad():
        final_outputs, _ = model(sequences)
        final_loss = criterion(final_outputs, targets)
    return final_loss.item()


def main():
    # Test different sequence lengths
    sequence_lengths = [5, 10, 20, 30, 50]
    results = {}

    print("Testing LSTM performance on different sequence lengths...")

    for seq_len in sequence_lengths:
        # Generate data and create model
        sequences, targets = generate_sequences(seq_len)
        model = SimpleLSTM(input_size=1, hidden_size=16, output_size=1)

        # Train and evaluate
        final_loss = train_and_evaluate(model, sequences, targets)
        results[seq_len] = final_loss

        print(f"Sequence length {seq_len}: Final loss = {final_loss:.4f}")

    # Visualize performance degradation
    plt.figure(figsize=(10, 6))
    lengths = list(results.keys())
    losses = list(results.values())

    plt.plot(lengths, losses, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Sequence Length')
    plt.ylabel('Final Loss (MSE)')
    plt.title('LSTM Performance Degradation with Increasing Sequence Length')
    plt.grid(True, alpha=0.3)
    plt.show()


if __name__ == "__main__":
    main()
```

---

### Solution

The completed `SimpleLSTM` class, together with the rest of the code, is below. Run it to see how the final loss changes with sequence length.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize hidden state and cell state
        h0 = torch.zeros(1, x.size(0), self.hidden_size)
        c0 = torch.zeros(1, x.size(0), self.hidden_size)

        # Forward propagate LSTM
        out, (hidden, cell) = self.lstm(x, (h0, c0))

        # Decode the hidden state of the last time step
        out = self.linear(out[:, -1, :])
        return out, hidden


def generate_sequences(seq_length, num_samples=100):
    """Generate sequences where target is the first element of the sequence"""
    torch.manual_seed(0)  # For reproducibility
    sequences = torch.randn(num_samples, seq_length, 1)
    targets = sequences[:, 0, 0]  # First element of each sequence
    return sequences, targets.unsqueeze(1)


def train_and_evaluate(model, sequences, targets, epochs=50):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs, _ = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    # Return final loss
    with torch.no_grad():
        final_outputs, _ = model(sequences)
        final_loss = criterion(final_outputs, targets)
    return final_loss.item()


def main():
    # Test different sequence lengths
    sequence_lengths = [5, 10, 20, 30, 50]
    results = {}

    print("Testing LSTM performance on different sequence lengths...")

    for seq_len in sequence_lengths:
        # Generate data and create model
        sequences, targets = generate_sequences(seq_len)
        model = SimpleLSTM(input_size=1, hidden_size=16, output_size=1)

        # Train and evaluate
        final_loss = train_and_evaluate(model, sequences, targets)
        results[seq_len] = final_loss

        print(f"Sequence length {seq_len}: Final loss = {final_loss:.4f}")

    # Visualize performance degradation
    plt.figure(figsize=(10, 6))
    lengths = list(results.keys())
    losses = list(results.values())

    plt.plot(lengths, losses, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Sequence Length')
    plt.ylabel('Final Loss (MSE)')
    plt.title('LSTM Performance Degradation with Increasing Sequence Length')
    plt.grid(True, alpha=0.3)
    plt.show()


if __name__ == "__main__":
    main()

## Generate Sequential Memory Challenge

Now that you've built the LSTM model in the previous exercise, it's time to create the data that will test its limits. Your job is to implement the `generate_sequences` function, which creates the challenging long-range dependency problem we want to study.

---

### Your Task

The function should generate random sequences in which the model needs to remember the very first element and predict it at the end. This means:

- creating random sequences of the specified length;
- using the first element of each sequence as the target;
- returning both sequences and targets in the proper tensor format;
- calling `torch.manual_seed()` to ensure reproducibility.

The provided code already has the training loop and visualization ready to go. Once you complete the data generation function, you'll be able to see how LSTM performance degrades as sequences get longer, giving you direct insight into why we need better architectures.

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # Initialize hidden state and cell state
        h0 = torch.zeros(1, x.size(0), self.hidden_size)
        c0 = torch.zeros(1, x.size(0), self.hidden_size)
        
        # Forward propagate LSTM
        out, (hidden, cell) = self.lstm(x, (h0, c0))
        
        # Decode the hidden state of the last time step
        out = self.linear(out[:, -1, :])
        return out, hidden

# TODO: Implement the generate_sequences function
# This function should:
# - Take seq_length and num_samples (default 100) as parameters
# - Generate random sequences where the target is the first element
# - Return sequences and targets in the correct format

def train_and_evaluate(model, sequences, targets, epochs=50):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs, _ = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    
    # Return final loss
    with torch.no_grad():
        final_outputs, _ = model(sequences)
        final_loss = criterion(final_outputs, targets)
    return final_loss.item()

def main():
    # Test different sequence lengths
    sequence_lengths = [5, 10, 20, 30, 50]
    results = {}
    
    print("Testing LSTM performance on different sequence lengths...")
    
    for seq_len in sequence_lengths:
        # Generate data and create model
        sequences, targets = generate_sequences(seq_len)
        model = SimpleLSTM(input_size=1, hidden_size=16, output_size=1)
        
        # Train and evaluate
        final_loss = train_and_evaluate(model, sequences, targets)
        results[seq_len] = final_loss
        
        print(f"Sequence length {seq_len}: Final loss = {final_loss:.4f}")
    
    # Visualize performance degradation
    plt.figure(figsize=(10, 6))
    lengths = list(results.keys())
    losses = list(results.values())
    
    plt.plot(lengths, losses, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Sequence Length')
    plt.ylabel('Final Loss (MSE)')
    plt.title('LSTM Performance Degradation with Increasing Sequence Length')
    plt.grid(True, alpha=0.3)
    plt.show()

if __name__ == "__main__":
    main()
```

---

### Solution

Only `generate_sequences` was missing:

- `sequences` has shape `(num_samples, seq_length, 1)`, i.e. `(batch, sequence, features)`, which is what `batch_first=True` expects.
- `targets` takes the first element of every sequence with `sequences[:, 0, 0]`; `.unsqueeze(1)` reshapes it to `(num_samples, 1)` to match the model output.
- `torch.manual_seed(0)` is called before `torch.randn`, so the data is reproducible.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # Initialize hidden state and cell state
        h0 = torch.zeros(1, x.size(0), self.hidden_size)
        c0 = torch.zeros(1, x.size(0), self.hidden_size)
        
        # Forward propagate LSTM
        out, (hidden, cell) = self.lstm(x, (h0, c0))
        
        # Decode the hidden state of the last time step
        out = self.linear(out[:, -1, :])
        return out, hidden

def generate_sequences(seq_length, num_samples=100):
    """Generate sequences where target is the first element of the sequence"""
    torch.manual_seed(0)  # For reproducibility
    sequences = torch.randn(num_samples, seq_length, 1)  # (batch, seq_len, features)
    targets = sequences[:, 0, 0]                         # First element of each sequence
    return sequences, targets.unsqueeze(1)               # targets -> (batch, 1)

def train_and_evaluate(model, sequences, targets, epochs=50):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs, _ = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    
    # Return final loss
    with torch.no_grad():
        final_outputs, _ = model(sequences)
        final_loss = criterion(final_outputs, targets)
    return final_loss.item()

def main():
    # Test different sequence lengths
    sequence_lengths = [5, 10, 20, 30, 50]
    results = {}
    
    print("Testing LSTM performance on different sequence lengths...")
    
    for seq_len in sequence_lengths:
        # Generate data and create model
        sequences, targets = generate_sequences(seq_len)
        model = SimpleLSTM(input_size=1, hidden_size=16, output_size=1)
        
        # Train and evaluate
        final_loss = train_and_evaluate(model, sequences, targets)
        results[seq_len] = final_loss
        
        print(f"Sequence length {seq_len}: Final loss = {final_loss:.4f}")
    
    # Visualize performance degradation
    plt.figure(figsize=(10, 6))
    lengths = list(results.keys())
    losses = list(results.values())
    
    plt.plot(lengths, losses, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Sequence Length')
    plt.ylabel('Final Loss (MSE)')
    plt.title('LSTM Performance Degradation with Increasing Sequence Length')
    plt.grid(True, alpha=0.3)
    plt.show()

if __name__ == "__main__":
    main()

## Switching Prediction Targets

## Training Your First LSTM Model